In [ ]:
import sisl
import numpy as np
from TimedependentTransport.Structures import structs
import matplotlib.pyplot as plt
from ase.visualize import view
from siesta_python.siesta_python import SiP
from GrainBoundaryModule.GrainBoundaryFunctions import more_screening, buffer_idx
import matplotlib.pyplot as plt
from TimedependentTransport.TimedependentTransport import TD_Transport
from TimedependentTransport.TimedependentTransport import AdaptiveRK4  as RK4
%matplotlib inline
plt.rcParams['figure.figsize'] = [15,10]

In [ ]:
i = 0#40      # 13,15 
d, em, ep, n = structs[i]
d, em, ep = more_screening(d, em, ep, 0,0)
bidx = buffer_idx(d, ep)
#view(d.toASE())
old_xyz = d.xyz.copy()

ky = 20
sisl.plot(d.tile(1,1))
plt.axis('equal')
#em = em.tile(2,0)


In [ ]:
eta = 1e-1
line = np.vstack([np.linspace(-3,3,65) + 1j * eta]*2)

G = TD_Transport([em, ep], d, kT_i = [0.025, 0.025])
G.Make_Contour(line, 15, pole_mode = 'JieHu2011')

G.Electrodes( semi_infs = ['-a1', '+a1'] , kp = [[40,ky,1],[40,ky,1]])
G.make_device(k = [1,ky,1], k_tbtrans = [1,5,1])
#G.run_electrodes()
#G.run_device()


In [ ]:
def buffer_cond(r):
    if (np.linalg.norm(r - old_xyz[bidx], axis = 1)<1e-2).any():
        return True
    return False
G.Device.set_buffer_atoms(buffer_cond)
G.Device.Visualise()

In [ ]:
G.run_electrodes()
G.run_device(morestuff = ['TS.Hartree.Fix -A'], where = 'TS_TBT')
G.read_data(sub_orbital = [4*i+2 for i in range(d.na - em.na - ep.na - len(G.Device.buffer_atoms))])

In [ ]:
G.Inspect_Transmission(0,1)


In [ ]:
#G.reset_all_fits() # Comment in to reset fits!
NL = 5
min_tol  = -0.0*np.ones(NL)
min_tol1 =  min_tol.copy()
min_tol2 =  min_tol.copy()

#ei1 = G.fitted_lorentzians[0].ei.copy(); gi1 = G.fitted_lorentzians[0].gamma.copy()
#ei1 = G.fitted_lorentzians[1].ei.copy(); gi1 = G.fitted_lorentzians[1].gamma.copy()


def run_mini(its):
    G.Fit(fact = 0.8, Fallback_W = 30.0, NumL = NL,
          fit_mode      = 'all',
          force_PSD     = True,
          force_PSD_tol = [min_tol1, min_tol2],
          use_analytical_jac = True,
          min_method = 'SLSQP',
          ebounds = (-4, 4),
          wbounds = (1e-6, 1.3),
          gbounds = (None, None),
          tol = -1,
          options = {'disp':True,'maxiter':its, 
                     'gtol':1e-10,
                     'ftol':1e-10,
                     'iprint':1
                     },
          fit_real_part = False,
          specific_bounds = None,#[{(0 ,2) :[(0.1, 0.11), (4,5)]}, {(0 ,5) :[(-0.1, 0.1), (4,5)]}], 
          alpha_PO = 0.0001, 
          )


In [ ]:
run_mini(50)

In [ ]:
#G.fitted_lorentzians[0].ei    = G.fitted_lorentzians[1].ei.copy()
#G.fitted_lorentzians[0].gamma[0] = G.fitted_lorentzians[1].gamma[0].copy()
G.pickle('GrainBoundary')


In [ ]:
print(G.fitted_lorentzians[1].is_zero)
IK = 2
G.Inspect_Lorentzian_fit(0,0,3,11,6,                          ik = IK, Emin = -7, Emax = 7, center_lines = True)


In [ ]:
G.Inspect_Poles(0)
plt.ylim([0,5])

In [ ]:
G.diagonalise()
G.get_propagation_quantities()
G.get_dense_matrices_purenp(1e-8)
G.Check_input_to_ODE(loose_fermi = True)
